# Romania 2001: acquire Arad and Sibiu gymnasium webpages

This notebook fetches **source webpages only** for Arad (AR) and Sibiu (SB). The saved Ministry directories list **136 AR** and **152 SB** originating-gymnasium webpages. Their county applicant, placement, unassigned, program-directory, and occupancy reports are already saved. A no-network preview found **0/288** gymnasium webpages saved at the start of this expansion.

The downloader reuses the source worker that completed CS, GL, and TL. Every successfully verified webpage is saved immediately in the private Desktop cache. Errors remain **unresolved** and retryable; they do not mean a school had zero applicants. The notebook does **not** calculate placement rates, peer strength, or HERO curves.

**Run order:** Cell 1 settings → Cell 2 load the worker → Cell 3 preview without network → Cell 4 researcher-controlled download → Cell 5 read progress. Choose the `sports_net` kernel in Cursor. Keep the analysis notebook and other Wayback download notebooks stopped while Cell 4 runs. See the [AR/SB acquisition plan](../outputs/romania_2001_county_expansion_20261004/acquisition_plan.md) for the source gate after downloading.

In [8]:
# CELL 1 — SETTINGS YOU CONTROL
# Re-run this cell after changing a setting. The download switch starts OFF.
from pathlib import Path

REPO_ROOT = Path("/Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE")
ANALYSIS_ROOT = REPO_ROOT / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
COUNTIES = ("AR", "SB")
RUN_DOWNLOAD = True

# Each verified page is checkpointed immediately. These caps bound one run;
# restart the notebook later to resume, rather than starting from page one.
MAX_NEW_PAGES = 300
MAX_HTTP_ATTEMPTS = 1000
MAX_HOURS = 12.0

# The worker adapts its pace, reports each request/wait, and responds to
# server pressure. These values preserve the previous successful policy.
MIN_INTERVAL_SECONDS = 22.0
MAX_INTERVAL_SECONDS = 50.0
FIRST_429_COOLDOWN_SECONDS = 600
SECOND_429_COOLDOWN_SECONDS = 1500
SERVER_ERROR_WAIT_SECONDS = 10.0

STATUS_FILE = ANALYSIS_ROOT / "outputs/romania_2001_county_expansion_20261004/gymnasium_acquisition_status.csv"
print("Counties:", ", ".join(COUNTIES))
print("Download enabled:", RUN_DOWNLOAD)
print("Per-request spacing:", MIN_INTERVAL_SECONDS, "to", MAX_INTERVAL_SECONDS, "seconds")
print("Name-free status file:", STATUS_FILE)
print("No network request has been made by this cell.")

Counties: AR, SB
Download enabled: True
Per-request spacing: 22.0 to 50.0 seconds
Name-free status file: /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/outputs/romania_2001_county_expansion_20261004/gymnasium_acquisition_status.csv
No network request has been made by this cell.


## Cell 2: load the source worker

This imports the Python worker but does **not** start downloading. It verifies that the selected counties are supported. The same private cache stores the older four-county source pages and any new AR/SB pages; the AR/SB **status CSV** is separate.

In [9]:
# CELL 2 — IMPORT ONLY; NO NETWORK REQUEST
import importlib
import sys

CODE_DIR = ANALYSIS_ROOT / "code"
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f"Code folder not found: {CODE_DIR}")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
worker = importlib.import_module("EDUCATION_20261002_romania_three_county_gymnasium_acquisition")
worker = importlib.reload(worker)
if not set(COUNTIES).issubset(set(worker.COUNTIES)):
    raise ValueError(f"Unsupported county in {COUNTIES}; worker supports {worker.COUNTIES}")
print("Loaded source-only worker:", worker.__file__)
print("Private saved-page folder:", worker.PAGES)
print("No Wayback request has been made.")

Loaded source-only worker: /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/code/EDUCATION_20261002_romania_three_county_gymnasium_acquisition.py
Private saved-page folder: /Users/charleslevine/Desktop/VECTOR_temp/romania_2001_four_county_school_source_pilot/school_reports
No Wayback request has been made.


## Cell 3: see what is already saved

This preview reads the saved Ministry school directories and private checkpoints. It makes **no Wayback request** and writes **no status file**. The first preview should show 0/136 AR and 0/152 SB unless acquisition has already started.

In [10]:
# CELL 3 — READ-ONLY PREVIEW; NO NETWORK REQUEST
worker.main([
    "--counties", *COUNTIES,
    "--status-file", str(STATUS_FILE),
])

2026-10-05 04:17:08 EDT | SOURCE-ONLY VIEW INVENTORY: 287/288 already verified for AR, SB; 1 still to obtain or retry
2026-10-05 04:17:08 EDT | PROGRESS AR: 135/136 gymnasium views verified; 1 unresolved; 0 not yet attempted
2026-10-05 04:17:08 EDT | PROGRESS SB: 152/152 gymnasium views verified; 0 unresolved; 0 not yet attempted
2026-10-05 04:17:08 EDT | THIS RUN: 0 newly saved pages; 0 school addresses attempted
2026-10-05 04:17:08 EDT | PREVIEW ONLY: no Wayback request and no status file written


## Cell 4: fetch only when you choose

Set `RUN_DOWNLOAD = True` in Cell 1 and rerun Cell 1, then run Cell 4. The worker prints the exact URL requested, each new verified page and row count, county progress, and the reason and expected end of a wait. It uses 600- then 1,500-second cooldowns for repeated HTTP 429 responses and stops after a third consecutive 429. Interrupting Cursor leaves all verified checkpoints intact. A later run retries unresolved school pages.

In [11]:
# CELL 4 — RESEARCHER-CONTROLLED SOURCE ACQUISITION
if not RUN_DOWNLOAD:
    print("Download is OFF. Change RUN_DOWNLOAD to True in Cell 1 and rerun Cell 1 first.")
else:
    run_args = [
        "--run", "--counties", *COUNTIES,
        "--status-file", str(STATUS_FILE),
        "--max-new-pages", str(MAX_NEW_PAGES),
        "--max-http-attempts", str(MAX_HTTP_ATTEMPTS),
        "--max-hours", str(MAX_HOURS),
        "--min-interval-seconds", str(MIN_INTERVAL_SECONDS),
        "--max-interval-seconds", str(MAX_INTERVAL_SECONDS),
        "--first-429-cooldown-seconds", str(FIRST_429_COOLDOWN_SECONDS),
        "--second-429-cooldown-seconds", str(SECOND_429_COOLDOWN_SECONDS),
        "--server-error-wait-seconds", str(SERVER_ERROR_WAIT_SECONDS),
    ]
    print("Starting SOURCE-ONLY acquisition for:", ", ".join(COUNTIES), flush=True)
    print("Expected directory-listed school webpages: AR 136, SB 152.", flush=True)
    print("Progress will be checkpointed after every requested school.", flush=True)
    worker.main(run_args)

Starting SOURCE-ONLY acquisition for: AR, SB
Expected directory-listed school webpages: AR 136, SB 152.
Progress will be checkpointed after every requested school.
2026-10-05 04:17:10 EDT | SOURCE-ONLY VIEW INVENTORY: 287/288 already verified for AR, SB; 1 still to obtain or retry
2026-10-05 04:17:10 EDT | PROGRESS AR: 135/136 gymnasium views verified; 1 unresolved; 0 not yet attempted
2026-10-05 04:17:10 EDT | PROGRESS SB: 152/152 gymnasium views verified; 0 unresolved; 0 not yet attempted
2026-10-05 04:17:10 EDT | THIS RUN: 0 newly saved pages; 0 school addresses attempted
2026-10-05 04:17:10 EDT | REQUEST AR gymnasium 190: https://web.archive.org/web/20020816151117id_/http://www.edu.ro/adm2001/raport_candidati_per_scoala.asp-cj=AR&nj=ARAD&cs=190&ns=SCOALA+CU+CLASELE+I-VIII+SOCODOR.htm
2026-10-05 04:17:11 EDT | WAIT 22.0s before next archive request; Client pacing before AR gymnasium 190 after robots.txt returned HTTP 404; 22.0s target without random variation; resume expected 2026-1

## Cell 5: read the saved progress

Run this after Cell 4 finishes or after an interruption. It reads the local, name-free status CSV. `saved_verified` means the school webpage and its identity checks passed. `unresolved` means a request still needs attention; it is **not** a zero-applicant school.

In [ ]:
# CELL 5 — LOCAL PROGRESS CHECK; NO NETWORK REQUEST
import pandas as pd
if not STATUS_FILE.exists():
    print("No AR/SB status file yet. Cell 3 is preview only; Cell 4 writes this file.")
else:
    progress = pd.read_csv(STATUS_FILE, dtype={"gymnasium_code": str}).fillna("")
    progress = progress.loc[progress["county"].isin(COUNTIES)].copy()
    totals = progress.groupby(["county", "status"], dropna=False).size().rename("school_webpages").reset_index()
    display(totals)
    for county in COUNTIES:
        local = progress.loc[progress["county"] == county]
        verified = int(local["status"].eq("saved_verified").sum())
        print(f"{county}: {verified}/{len(local)} gymnasium webpages verified")
    unresolved = progress.loc[progress["status"].str.startswith("unresolved")]
    print("Unresolved school webpages:", len(unresolved))
    if not unresolved.empty:
        display(unresolved[["county", "gymnasium_code", "status", "last_http_status"]])
    print("Status file:", STATUS_FILE)

## What happens after downloading

We will run an **offline source reconciliation** before using either county in a plot. That check must compare originating-gymnasium applicants with saved county applicant, placement, and unassigned reports; verify exam and grade components and program identities; and list cross-county placements and unresolved person matches. A saved webpage is a verified source checkpoint, not proof that every eighth grader participated. No AR/SB HERO or six-county result follows automatically from this notebook.